In [ ]:
!pip install sentence-transformers --quiet

# Dependencies

import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

## Descriptions

In [ ]:
descriptions = {
    # Earthquakes: same event, different vocabulary
    'A': 'A magnitude 7.1 earthquake struck offshore, causing violent ground shaking in coastal cities.',
    'B': 'Strong seismic activity was recorded near the coast, with significant shaking felt across the region.',
    'C': 'A large rupture along a subduction zone fault released enormous energy, shaking the surrounding area.',
    # Tsunamis: causally linked to earthquakes
    'D': 'A destructive ocean wave, several meters high, inundated low-lying coastal communities.',
    'E': 'Following the seismic event, a series of large waves crossed the ocean and flooded the shoreline.',
    # Volcanic eruptions
    'F': 'The volcano erupted explosively, sending a column of ash and gas high into the atmosphere.',
    'G': 'Lava flows advanced slowly downslope, destroying vegetation and infrastructure in their path.',
    # Wildfire: superficially similar to volcanic eruption
    'H': 'A rapidly spreading wildfire sent thick smoke into the atmosphere, reducing air quality across the region.',
    # Flooding: causally linked to heavy rainfall
    'I': 'Prolonged heavy rainfall caused rivers to overflow their banks, flooding towns in the valley below.',
    'J': 'Floodwaters inundated low-lying areas after an unusually wet season, displacing thousands of residents.',
    # Landslides: causally linked to rainfall
    'K': 'Saturated soils on a steep hillside gave way, sending a debris flow into the valley below.',
    'L': 'Heavy rain destabilized a hillside, triggering a mass movement that buried a road.',
    # Hurricane / tropical storm
    'M': 'A powerful tropical cyclone made landfall, bringing extreme winds, storm surge, and heavy rainfall.',
    'N': 'The hurricane caused widespread destruction along the coast due to high winds and coastal flooding.',
    # Drought
    'O': 'An extended period of below-average precipitation led to severe water shortages and crop failure.',
    # Avalanche: similar trigger mechanism to landslide
    'P': 'Unstable snowpack on a steep mountain slope released suddenly, burying a trail below.',
}

labels = list(descriptions.keys())
sentences = list(descriptions.values())

## Embeddings

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(sentences)

print(f'Each sentence is a vector of {embeddings.shape[1]} numbers.')
print(f'{embeddings.shape[0]} sentences embedded.')

## Similarity matrix

In [ ]:
# Generate a similarity matrix

sim_matrix = cosine_similarity(embeddings)

In [ ]:
# Plot the pairwise cosine similiarity

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine similarity')
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=11)
ax.set_yticklabels(labels, fontsize=11)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=7)
ax.set_title('Pairwise cosine similarity: natural hazard descriptions', fontsize=13)
plt.tight_layout()
plt.show()

## Finding the nearest neighbors

In [ ]:
# Show us which descriptions are closest to each description.

for i, label in enumerate(labels):
    sims = [(labels[j], sim_matrix[i,j]) for j in range(len(labels)) if j != i]
    sims_sorted = sorted(sims, key=lambda x: x[1], reverse=True)
    print(f'\n--- {label}: {sentences[i]}')
    for neighbor, score in sims_sorted[:3]:
        print(f'  {neighbor} ({score:.3f}): {descriptions[neighbor]}')

## A question

### Does the embedding understand the same event if it is described differently?

Below are two descriptions of natural hazard events written in ways that differ from the originals —
one uses technical geoscience language, the other describes the human experience rather than the physical process.

**Before running any code**, write down:
- Which of the original descriptions (A–P) you think each one is most similar to and why.

Then run the code cell and compare the embedding's answer to your prediction.



**Description X** (technical language):

*"A Mw 7.1 interplate megathrust rupture occurred on the subduction interface, generating strong
ground motion with peak ground accelerations exceeding 0.3g in nearby urban areas."*

Your prediction: _______ because: _______



**Description Y** (consequence / human impact language):

*"Several skiers were reported missing after being buried under snow on a backcountry slope
following a loud rumbling sound."*

Your prediction: _______ because: _______

In [ ]:
new_descriptions = {
    'X': 'A Mw 7.1 interplate megathrust rupture occurred on the subduction interface, generating strong ground motion with peak ground accelerations exceeding 0.3g in nearby urban areas.',
    'Y': 'Several skiers were reported missing after being buried under snow on a backcountry slope following a loud rumbling sound.',
}

new_embeddings = model.encode(list(new_descriptions.values()))
new_sims = cosine_similarity(new_embeddings, embeddings)

for i, (label, desc) in enumerate(new_descriptions.items()):
    print(f'\n--- {label}: {desc}')
    print('  Most similar originals:')
    ranked = sorted(zip(labels, new_sims[i]), key=lambda x: x[1], reverse=True)
    for neighbor, score in ranked[:5]:
        print(f'  {neighbor} ({score:.3f}): {descriptions[neighbor]}')

### Written response

1. Did the embedding agree with your predictions? For each of X and Y, explain why or why not.

2. Compare the similarity scores for X against A, B, C to the within-group similarities
you found in Question 1. Is X as close to the earthquake descriptions as they are to each other?
What might explain any gap?

3. For Y: the description says nothing about snow instability, slopes, or mass movement —
only about people being buried after a sound. Does the embedding still find the avalanche (P)?
What does this tell you about what the model has learned — and what it might be missing?

**Provide your answers to these questions as a single .md file in this folder.**

## Bonus: Dimensional Reduction



384 dimensions are hard to visualize! 

What if we use PCA to reduce dimensionality?

In [ ]:
pca = PCA(n_components=2)
emb2d = pca.fit_transform(embeddings)

var_explained = pca.explained_variance_ratio_ * 100
print(f'PC1 explains {var_explained[0]:.1f}% of variance')
print(f'PC2 explains {var_explained[1]:.1f}% of variance')
print(f'Total: {var_explained.sum():.1f}%')

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(emb2d[:,0], emb2d[:,1], s=80, color='steelblue', zorder=3)
for i, label in enumerate(labels):
    ax.annotate(f"{label}: {sentences[i][:45]}...",
                (emb2d[i,0], emb2d[i,1]),
                fontsize=7, xytext=(6,4), textcoords='offset points')
ax.set_title('2D PCA projection of natural hazard embeddings', fontsize=12)
ax.set_xlabel(f'PC1 ({var_explained[0]:.1f}% variance)')
ax.set_ylabel(f'PC2 ({var_explained[1]:.1f}% variance)')
plt.tight_layout()
plt.show()